# What time is it? A CNN reads a sunrise

On the slides, one photo of a sunrise answers two questions: the **season**, a classification, and the **time of the day**, a regression. This notebook answers the second one. A small **convolutional neural network**, a CNN as on weekend 3, looks at a sunrise and says what time it is.

The pictures are synthetic: they are drawn by the generator of the Sunrise Machine, the exercise of 09:00, written by Frederick Puls. A seeded random generator picks the moments, so every run sees the same sunrises.

**Your part: four tasks, each marked with 🎯, seven lines in all.** The two convolution blocks of the body and the head in part 3, the loss and the three calls of one training step in part 4. Everything else is done for you. The cells that show only a title make the pictures, draw them and score the network: run them, there is nothing to change inside. Their Show code button opens one if you are curious.

**Stuck?** Every task has a hint below it: click it to open it. And the season classifier is the worked example: the same pictures and the same body, with another head and another loss, and all of its code shown:

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/sunrise-cnn/sunrise_season_classifier.ipynb

In [ ]:
#@title Imports { display-mode: "form" }
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

## 1. The sunrises

A picture is $32 \times 32$ pixels in colour, so $32 \times 32 \times 3 = 3072$ numbers between $0$ and $1$: how red, how green and how blue each pixel is. The generator draws it from two numbers between $0$ and $1$:

* `sun_x`, where the sun stands, from left to right: the **time of the year**, from winter at $0$ to summer at $1$,
* `sun_y`, how high the sun stands: the **time of the day**, from dawn at $0$ to noon at $1$.

The colours of the sky follow the sun: indigo and gold at dawn, blue at noon, a little warmer in summer. The hills on the horizon change with the time of the year too.

In [ ]:
#@title The Sunrise Machine's generator, by Frederick Puls { display-mode: "form" }
# Written by Frederick Puls for the Sunrise Machine, the exercise of 09:00, in its file
# generate_dataset.py, and copied here with three changes that leave every picture the
# same: its loops run over lists instead of over a range, its comments are plain ASCII
# text, and its type hints are gone.
#
# render_image(sun_x, sun_y) draws one sunrise, 32 x 32 pixels in colour, from
#   sun_x, where the sun stands from left to right: the time of the year, winter 0 to summer 1,
#   sun_y, how high the sun stands: the time of the day, dawn 0 to noon 1.
import math


# ---------------------------------------------------------------------------
# Reference sky palettes: each is a list of (row_fraction, RGB) color stops.
# We define four palettes keyed by sun_y, and interpolate between them.
# ---------------------------------------------------------------------------

# Dawn  (sun_y about 0)
_PALETTE_DAWN = np.array([
    [0.08, 0.04, 0.18],  # top: dark indigo
    [0.20, 0.08, 0.28],  # deep violet
    [0.70, 0.25, 0.35],  # rose
    [0.90, 0.60, 0.20],  # gold at horizon
])

# Early morning (sun_y about 0.3)
_PALETTE_EARLY = np.array([
    [0.05, 0.05, 0.20],  # navy
    [0.15, 0.25, 0.55],  # blue
    [0.85, 0.60, 0.45],  # peach
    [0.90, 0.50, 0.15],  # orange at horizon
])

# Mid morning (sun_y about 0.7)
_PALETTE_MID = np.array([
    [0.15, 0.30, 0.65],  # blue
    [0.40, 0.60, 0.85],  # light blue
    [0.80, 0.80, 0.60],  # pale yellow
    [0.90, 0.85, 0.55],  # yellow at horizon
])

# Noon (sun_y about 1.0)
_PALETTE_NOON = np.array([
    [0.10, 0.20, 0.55],  # deep blue
    [0.30, 0.55, 0.85],  # sky blue
    [0.70, 0.80, 0.95],  # light sky
    [0.90, 0.90, 0.92],  # white at horizon
])

_PALETTES = np.stack([_PALETTE_DAWN, _PALETTE_EARLY, _PALETTE_MID, _PALETTE_NOON])
_PALETTE_Y_KEYS = np.array([0.0, 0.3, 0.7, 1.0])

# Row fractions where each colour stop sits (same for every palette)
_STOP_ROWS = np.array([0.0, 0.33, 0.66, 1.0])


def _interpolate_palette(sun_y, sun_x):
    """Return a (4, 3) palette interpolated for the given sun_y, then
    warm-shifted by sun_x."""
    # Piecewise-linear interp between the 4 reference palettes
    idx = np.searchsorted(_PALETTE_Y_KEYS, sun_y, side="right") - 1
    idx = np.clip(idx, 0, len(_PALETTE_Y_KEYS) - 2)
    t = (sun_y - _PALETTE_Y_KEYS[idx]) / (_PALETTE_Y_KEYS[idx + 1] - _PALETTE_Y_KEYS[idx])
    palette = (1 - t) * _PALETTES[idx] + t * _PALETTES[idx + 1]

    # sun_x warmth shift: push towards orange/gold, boost saturation
    warm_shift = np.array([0.08, 0.02, -0.06])  # add red/green, remove blue
    palette = palette + warm_shift * (sun_x - 0.5) * 2  # centred at 0.5
    return np.clip(palette, 0, 1)


def render_image(sun_x, sun_y, size=32):
    """Render a single sunrise image as (size, size, 3) float32 in [0,1]."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    palette = _interpolate_palette(sun_y, sun_x)

    # ---- 1. Sky gradient ----
    row_fracs = np.linspace(0, 1, size).reshape(-1, 1)  # (H, 1)
    # Interpolate across the 4 colour stops
    for i in (0, 1, 2):
        mask = (row_fracs >= _STOP_ROWS[i]) & (row_fracs <= _STOP_ROWS[i + 1])
        t = (row_fracs - _STOP_ROWS[i]) / (_STOP_ROWS[i + 1] - _STOP_ROWS[i])
        blend = (1 - t) * palette[i] + t * palette[i + 1]  # (H, 3)
        # broadcast across width
        for c in (0, 1, 2):
            ch = blend[:, c:c+1] * np.ones((1, size))
            img[:, :, c] = np.where(mask, ch, img[:, :, c])

    # ---- 2. Sun disc (Gaussian blob) ----
    sx = sun_x * size
    sy = (1 - sun_y) * size * 0.7 + 0.1 * size
    radius = size * 0.12  # ~3.8 px at 32

    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    dist2 = (xx - sx) ** 2 + (yy - sy) ** 2
    sun_blob = np.exp(-dist2 / (2 * radius ** 2))

    # Sun colour depends on elevation
    if sun_y < 0.3:
        sun_color = np.array([1.0, 0.45, 0.15])  # deep orange
    elif sun_y < 0.7:
        t = (sun_y - 0.3) / 0.4
        sun_color = (1 - t) * np.array([1.0, 0.65, 0.2]) + t * np.array([1.0, 0.9, 0.5])
    else:
        t = (sun_y - 0.7) / 0.3
        sun_color = (1 - t) * np.array([1.0, 0.9, 0.5]) + t * np.array([1.0, 0.98, 0.85])

    for c in (0, 1, 2):
        img[:, :, c] = img[:, :, c] + sun_blob * sun_color[c] * 1.2

    # ---- 3. Atmospheric glow ----
    glow_radius = size * 0.5
    glow = np.exp(-dist2 / (2 * glow_radius ** 2))
    glow_intensity = 0.4 * max(0, 1 - sun_y * 1.5)  # stronger at low sun
    glow_color = np.array([0.9, 0.55, 0.15])
    for c in (0, 1, 2):
        img[:, :, c] = img[:, :, c] + glow * glow_color[c] * glow_intensity

    # ---- 4. Horizon silhouette ----
    horizon_base = int(size * 0.85)
    # gentle hills via sine
    phase = sun_x * 2 * math.pi
    hill_y = np.sin(np.linspace(0, 2 * math.pi, size) + phase) * size * 0.03
    for col, hill in enumerate(hill_y):
        h = int(horizon_base + hill)
        h = max(0, min(size - 1, h))
        # dark land colour: blend dark blue-black to dark green-brown by sun_x
        land_a = np.array([0.04, 0.04, 0.10])
        land_b = np.array([0.10, 0.16, 0.10])
        land_color = (1 - sun_x) * land_a + sun_x * land_b
        img[h:, col, :] = land_color

    # ---- 5. Noise & clip ----
    img += np.random.normal(0, 0.01, img.shape).astype(np.float32)
    np.clip(img, 0, 1, out=img)

    return img

## 2. Three thousand sunrises, and their times

The same $3000$ moments as in the season classifier, picked at random, and a sunrise for each. The network learns from the first $2400$, the **training** sunrises. The last $600$ are kept apart as **test** sunrises: the network never sees them while it learns, so they show how well it does on pictures it has not seen.

The time of the day comes from the height of the sun. It touches the horizon at $6$ o'clock and stands highest at noon, so the time of a sunrise in hours is $y = 6 + 6 \cdot$ `sun_y`, between $6$ and $12$: $7.5$ is half past seven. Real dawns come earlier in summer than in winter; the generator leaves that out, and so do we.

In [ ]:
#@title Three thousand sunrises { display-mode: "form" }
np.random.seed(0)   # seeded: the same moments, and the same pictures, on every computer

# 3000 moments, one row (sun_x, sun_y) each, both between 0 and 1, and a sunrise for each.
sun = np.random.uniform(0, 1, size=(3000, 2))
pictures = np.array([render_image(sun_x, sun_y) for sun_x, sun_y in sun])   # 3000 x 32 x 32 x 3
pictures = torch.tensor(pictures).permute(0, 3, 1, 2).contiguous()        # PyTorch wants the colours first

# The time of the day in hours, from 6 at dawn to 12 at noon: 7.5 is half past seven.
hours = torch.tensor(6 + 6 * sun[:, 1], dtype=torch.float32).reshape(3000, 1)

# The first 2400 sunrises to train on, the last 600 to test on.
train_pictures, test_pictures = pictures[:2400], pictures[2400:]
train_hours, test_hours = hours[:2400], hours[2400:]
print(len(train_pictures), "training sunrises and", len(test_pictures), "test sunrises,",
      "each", tuple(train_pictures.shape[1:]), "numbers")


def clock(hour):
    # 7.5 hours as "7:30"
    minutes = round(60 * float(hour))
    return f"{minutes // 60}:{minutes % 60:02d}"

In [ ]:
#@title Twelve sunrises and their times { display-mode: "form" }
def show(ax, picture, title):
    ax.imshow(picture.permute(1, 2, 0))   # the colours last again, for drawing
    ax.set(xticks=[], yticks=[], title=title)



# Twelve training sunrises, from the earliest to the latest.
picks = train_hours[:12, 0].argsort()
fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for ax, i in zip(axes.flat, picks):
    show(ax, train_pictures[i], clock(train_hours[i]))
plt.show()

## 3. The network

The same network as in the season classifier, with one change at the end. It has two parts.

* Its **body** reads the picture with two **convolution blocks**, one line each. `nn.Conv2d` slides small filters of $3 \times 3$ pixels over the picture, as on weekend 3, and each filter makes a new picture, a **map**; `padding=1` keeps the map as wide and as high as the picture. `nn.ReLU()` turns every negative number into $0$. `nn.MaxPool2d(2)` keeps the largest number of every square of $2 \times 2$, which halves the width and the height. The first block turns the $3$ colours of $32 \times 32$ pixels into $8$ maps of $16 \times 16$, the second turns those into $16$ maps of $8 \times 8$. Then `nn.Flatten()` lays them out as one list of $16 \times 8 \times 8 = 1024$ numbers.
* Its **head** turns the $1024$ numbers into the answer: **one** number, the predicted time in hours, $\hat{y}$.

🎯 **Task 1.** Write the two convolution blocks of the body, one line each:

* the first takes the $3$ colours of the picture to $8$ maps, with filters of $3 \times 3$ pixels and `padding=1`, followed by an `nn.ReLU()` and an `nn.MaxPool2d(2)`,
* the second takes those $8$ maps to $16$ maps, the same way.

<details>
<summary><b>Hint for task 1</b> (click to open)</summary>

The season classifier has the same body, line for line: in its part 3, the two lines under `body = nn.Sequential(` are the two lines asked for here. A line reads

`nn.Conv2d(maps in, maps out, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),`

and the maps out of the first line are the maps in of the second.

</details>

🎯 **Task 2.** Write the head: one `nn.Linear` that takes the $16 \times 8 \times 8 = 1024$ numbers of the body to **one** number, the predicted time.

<details>
<summary><b>Hint for task 2</b> (click to open)</summary>

In part 3 of the season classifier the head is `nn.Linear(16 * 8 * 8, 3)`: $1024$ numbers in, $3$ scores out, one per season. Here the answer is a single number.

</details>

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

# ============================================================
# 🎯 TASK 1: the two convolution blocks of the body
# ============================================================
# The body reads the picture: two convolution blocks, one line each.
body = nn.Sequential(
    # 🎯 COMPLETE THIS LINE: 3 colours into 8 maps, 3 x 3 filters, padding 1, then ReLU and max pooling
    ____,
    # 🎯 COMPLETE THIS LINE: 8 maps into 16 maps, the same way
    ____,
    nn.Flatten(),                                                             # 16 x 8 x 8 = 1024 numbers in one list
)

# ============================================================
# 🎯 TASK 2: the head
# ============================================================
# The head gives the answer: 1 number, the predicted time in hours.
# 🎯 COMPLETE THIS LINE: an nn.Linear from the 16 x 8 x 8 = 1024 numbers of the body to 1 number
head = ____

model = nn.Sequential(body, head)

## 4. Training

The loop of weekend 1, with the loss of the slides. The miss of one picture is the squared difference between its true time $y_i$ and the predicted one $\hat{y}_i$, $\mathscr{L}_i = (y_i - \hat{y}_i)^2$, and the loss $\bar{\mathscr{L}}$ is its average over $64$ training sunrises picked at random. The times are in hours, so the loss is in hours squared: the loop stops once it is below $0.0025$, a typical miss of $0.05$ hours, which is $3$ minutes, since $0.05 \times 0.05 = 0.0025$. Under the cell, the typical miss of the last $64$ sunrises is printed in minutes as it falls.

🎯 **Task 3.** Write the loss: the average squared miss of the $64$ predicted times `guess` against the true times `train_hours[batch]`, the $\bar{\mathscr{L}}$ above.

<details>
<summary><b>Hint for task 3</b> (click to open)</summary>

The season classifier's loss, `nn.functional.cross_entropy`, is made for choosing one of several classes, so it does not fit a time. Measure the miss as on the slides: subtract the true times from the predicted ones, square the differences with `** 2`, and average them with `.mean()`.

</details>

🎯 **Task 4.** One training step is three calls, always in this order:

1. The optimizer forgets the gradients of the previous step.
2. The loss computes its gradient, backwards through the network.
3. The optimizer moves every weight a small step against its gradient.

<details>
<summary><b>Hint for task 4</b> (click to open)</summary>

The loop of the season classifier, in its part 4, ends with the same three calls, in the same order: copy them.

</details>

On Colab the loop stopped after about $20$ seconds. It stops by itself once the typical miss of a batch is under $3$ minutes. If the typical miss printed under the cell is still above $60$ minutes after half a minute, stop the cell with the square button next to it and check your lines.

In [ ]:
# ============================================================
# 🎯 TASKS 3 AND 4: the loss and one training step, in the loop below
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)   # lr: how big a step is

loss = 1.0                                                     # any start above the threshold
while loss > 0.0025:
    batch = torch.randint(0, len(train_pictures), (64,))       # 64 training sunrises picked at random
    guess = model(train_pictures[batch])                       # the predicted time of each of them
    # 🎯 TASK 3, COMPLETE THIS LINE: the average squared miss of guess against the true times
    loss = ____
    # 🎯 TASK 4, COMPLETE THESE THREE LINES: one training step
    ____    # 1. the optimizer forgets the gradients of the previous step
    ____    # 2. the loss computes its gradient, backwards through the network
    ____    # 3. the optimizer moves every weight a small step against its gradient
    print(f"\raverage squared miss: {loss.item():9.4f}, a typical miss of {60 * loss.item() ** 0.5:6.1f} minutes", end="")

## 5. How close is it?

The folded cell below asks the network for the time of the $600$ test sunrises and prints how far off it is, in minutes. In the plot every test sunrise is a dot, at its true time across and its predicted time up: a perfect network would put every dot on the diagonal line. Last come twelve of the test sunrises, with the true and the predicted time.

In [ ]:
#@title The time of the 600 test sunrises { display-mode: "form" }
with torch.no_grad():                        # no training here, so no gradients
    guess = model(test_pictures)             # the predicted time of every test sunrise
miss = 60 * (guess - test_hours).abs()       # how far off, in minutes
print(f"average miss on the {len(miss)} test sunrises: {miss.mean():.1f} minutes")
print(f"largest miss: {miss.max():.1f} minutes")

ticks = [6, 7, 8, 9, 10, 11, 12]
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot([6, 12], [6, 12], color="#D55E00", linewidth=1.5, label="a perfect guess")
ax.scatter(test_hours, guess, s=10, color="#0072B2", label="a test sunrise")
ax.set(xlim=(5.8, 12.2), ylim=(5.8, 12.2), xticks=ticks, yticks=ticks,
       xticklabels=[clock(hour) for hour in ticks], yticklabels=[clock(hour) for hour in ticks],
       xlabel="true time", ylabel="predicted time")
ax.legend(loc="upper left")
plt.show()

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, picture, true, said in zip(axes.flat, test_pictures, test_hours, guess):
    show(ax, picture, f"true {clock(true)}\nsaid {clock(said)}")
plt.show()